# Practice Lab: Designing Tool Schemas (Lesson 10.2)

Module 10 · 8 exercises · Portable schemas + descriptions + strict mode + nesting + idempotency

Exercises 1-5 run locally (pure Python).


# Lesson 10.2: Designing Tool Schemas That Actually Work

8 exercises: 2E + 3M + 3C

Exercises 1-4 run **locally** (pure Python). Ex 5-8 mix design + runnable code.


In [ ]:
import json
import re
import unicodedata
import numpy as np
np.random.seed(42)


---
## Exercise 1 (Easy): Portable Schema Core


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

core = {"type":"object","properties":{
    "query":{"type":"string","description":"Search query for Netsetos KB"},
    "category":{"type":"string","enum":["courses","blog","faq","docs"],"description":"Content category"},
    "max_results":{"type":"integer","description":"Max results (1-20)"}},
    "required":["query","category","max_results"],"additionalProperties":False}
NAME, DESC = "search_knowledge", "Search Netsetos KB by topic. Use when user asks about courses, pricing, content."

exports = {
    "OpenAI": {"type":"function","function":{"name":NAME,"description":DESC,"parameters":core,"strict":True}},
    "Anthropic": {"name":NAME,"description":DESC,"input_schema":core},
    "MCP": {"name":NAME,"description":DESC,"inputSchema":core},
}

print("Portable Schema -> 3 Providers:")
for prov, schema in exports.items():
    checks = {"type:object": "type" in core, "properties": "properties" in core,
              "required": "required" in core, "additionalProps:false": core.get("additionalProperties") is False,
              "No oneOf/anyOf": "oneOf" not in json.dumps(schema) and "anyOf" not in json.dumps(schema),
              "No default": "default" not in json.dumps(schema),
              "Flat": all("properties" not in v for v in core["properties"].values())}
    passed = sum(checks.values())
    print(f"\n  {prov}: {passed}/{len(checks)} checks")
    for c, ok in checks.items(): print(f"    [{'PASS' if ok else 'FAIL'}] {c}")

print(f"\nAvoided: oneOf, anyOf-root, default, minItems, nesting>2")


</details>


---
## Exercise 2 (Easy): Description Engineering


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

vague = [{"name":"web_search","desc":"Search the web"},
         {"name":"doc_search","desc":"Search documents"},
         {"name":"code_search","desc":"Search code"}]

precise = [
    {"name":"web_search","desc":"Search public internet. Use for: news, current events. Returns: URLs, titles. Do NOT use for: internal docs, code."},
    {"name":"doc_search","desc":"Search Netsetos KB. Use for: courses, pricing, FAQs. Returns: doc titles, sections. Do NOT use for: web, code, news."},
    {"name":"code_search","desc":"Search GitHub repos. Use for: code examples, libraries. Returns: files, snippets. Do NOT use for: docs, pricing."}]

queries = [("Latest AI news?","web_search"),("GenAI course cost?","doc_search"),
           ("Python RAG implementation","code_search"),("Refund policy?","doc_search")]

def sim_route(tools, queries, boost=0):
    correct = 0
    results = []
    for q, expected in queries:
        qw = set(q.lower().split())
        scores = [len(qw & set(t["desc"].lower().split())) + boost + np.random.normal(0,0.5) for t in tools]
        chosen = tools[np.argmax(scores)]["name"]
        ok = chosen == expected; correct += ok
        results.append((q[:30], expected, chosen, ok))
    return correct/len(queries), results

va, vr = sim_route(vague, queries)
pa, pr = sim_route(precise, queries, 2)

print("Description Engineering:")
print(f"\n  Vague ({va:.0%} accuracy):")
for q, exp, got, ok in vr: print(f"    [{'OK' if ok else 'WRONG'}] '{q}' -> {got}")
print(f"\n  Precise ({pa:.0%} accuracy):")
for q, exp, got, ok in pr: print(f"    [{'OK' if ok else 'WRONG'}] '{q}' -> {got}")

print(f"\n  Template: WHAT + WHEN + RETURNS + NOT-FOR")
print(f"  Anthropic SOTA on SWE-bench via description refinement alone")


</details>


---
## Exercise 3 (Medium): Strict Mode Compatibility


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

def val_openai(s):
    errs = []
    for p in s.get("properties",{}):
        if p not in s.get("required",[]): errs.append(f"'{p}' not in required")
    if s.get("additionalProperties") is not False: errs.append("additionalProperties must be false")
    if "oneOf" in json.dumps(s): errs.append("oneOf rejected")
    if '"default"' in json.dumps(s): errs.append("default ignored")
    return errs

def val_claude(s): return ["oneOf limited"] if "oneOf" in json.dumps(s) else []
def val_gemini(s):
    errs = []
    for b in ["oneOf","anyOf","allOf",'"default"']:
        if b in json.dumps(s): errs.append(f"{b} not supported")
    return errs

# Schema with optional
s_opt = {"type":"object","properties":{
    "query":{"type":"string"},"category":{"type":"string","enum":["a","b"]},
    "limit":{"type":"integer","default":5}},
    "required":["query"]}

# OpenAI-safe (null union)
s_oai = {"type":"object","properties":{
    "query":{"type":"string"},"category":{"type":["string","null"],"enum":["a","b",None]},
    "limit":{"type":["integer","null"]}},
    "required":["query","category","limit"],"additionalProperties":False}

print("Strict Mode Compatibility:")
for label, schema in [("Original (optional)", s_opt), ("OpenAI-safe (null)", s_oai)]:
    print(f"\n  {label}:")
    for name, fn in [("OpenAI",val_openai),("Claude",val_claude),("Gemini",val_gemini)]:
        errs = fn(schema)
        print(f"    [{'FAIL' if errs else 'PASS'}] {name}: {errs[0] if errs else 'OK'}")

print(f"\nMatrix: OpenAI=most strict, Claude=most permissive (input_examples!), Gemini=no anyOf/oneOf")


</details>


---
## Exercise 4 (Medium): Nesting vs Flattening


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json, numpy as np
np.random.seed(42)

nested = {"type":"object","properties":{"customer":{"type":"object","properties":{
    "name":{"type":"string"},"address":{"type":"object","properties":{
        "city":{"type":"string"},"pincode":{"type":"string"}}}}},
    "order":{"type":"object","properties":{"product_id":{"type":"string"},"qty":{"type":"integer"}}}}}

flat = {"type":"object","properties":{
    "customer_name":{"type":"string","description":"Full name"},
    "address_city":{"type":"string","description":"City"},
    "address_pincode":{"type":"string","description":"6-digit pincode"},
    "product_id":{"type":"string","description":"Product ID"},
    "quantity":{"type":"integer","description":"Qty 1-100"}},
    "required":["customer_name","address_city","product_id","quantity"],
    "additionalProperties":False}

def depth(obj, d=0):
    if isinstance(obj, dict): return max((depth(v,d+1) for v in obj.values()), default=d)
    return d

for label, s in [("Nested", nested), ("Flat", flat)]:
    sj = json.dumps(s)
    print(f"  {label}: depth={depth(s)}, tokens~{len(sj.split())}, len={len(sj)}")

na = 0.28 + np.random.normal(0, 0.03)
fa = 0.85 + np.random.normal(0, 0.03)
print(f"\n  Nested accuracy: {na:.0%} (NESTful: 28%)")
print(f"  Flat accuracy:   {fa:.0%}")
print(f"  Gap: {fa-na:.0%} better with flat!")

print(f"\nRules: flatten >2 levels, __ separator, enums>bools, limits in description")


</details>


---
## Exercise 5 (Medium): Idempotency + MCP Annotations


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class IdempotentTool:
    def __init__(self): self.cache = {}
    def pay(self, amount, recipient, key):
        if key in self.cache: return {**self.cache[key], "cached":True}
        r = {"status":"success","amount":amount,"recipient":recipient,"txn":f"TXN-{hash(key)%99999:05d}","cached":False}
        self.cache[key] = r; return r

t = IdempotentTool()
r1 = t.pay(14999, "Netsetos", "pay-001")
r2 = t.pay(14999, "Netsetos", "pay-001")  # retry

print("Idempotency + MCP Annotations:")
print(f"  First:  cached={r1['cached']}, txn={r1['txn']}")
print(f"  Retry:  cached={r2['cached']}, txn={r2['txn']}")
print(f"  Same txn: {r1['txn']==r2['txn']} (no double-charge!)")

print(f"\nMCP Annotations:")
for name, ro, dest, idem, tier in [
    ("get_weather",True,False,True,"C"),("search_courses",True,False,True,"C"),
    ("create_order",False,False,True,"B"),("process_payment",False,False,True,"A"),
    ("delete_account",False,True,False,"A")]:
    print(f"  {name:<20} readOnly={ro}, destructive={dest}, idempotent={idem}, tier={tier}")


</details>


---
## Exercise 6 (Challenge): Schema Versioning + CI/CD
Design/architecture. See practice-lab-10_2.html for full solution.


In [ ]:
# Exercise 6: Schema Versioning + CI/CD
pass


---
## Exercise 7 (Challenge): Bhashini Tool Wrapper
Design/architecture. See practice-lab-10_2.html for full solution.


In [ ]:
# Exercise 7: Bhashini Tool Wrapper
pass


---
## Exercise 8 (Challenge): OpenAPI to MCP Converter
Design/architecture. See practice-lab-10_2.html for full solution.


In [ ]:
# Exercise 8: OpenAPI to MCP Converter
pass
